# Day 7 · Lab 2: Building a Custom MCP Server in Python
### Production AI Frameworks Curriculum · Enterprise AI Engineering · Udaan Learning

---

## 🎯 Learning Objectives
1. Implement a complete Python **MCP Server** using standard decorators and handlers.
2. Define parameterized **Tools** with Pydantic / JSON Schema validation.
3. Expose dynamic **Resources** using URI templates (`ehr://patient/{mrn}`).
4. Build an interactive **Prompt Template** for clinical chart reconciliation.
5. Test the server using simulated JSON-RPC requests.



In [ ]:
import json
import sqlite3
import re
from typing import Dict, Any, List, Optional

print("[*] Lab 2: Custom MCP Server environment ready.")


---
## 1. Setting up the Server Backend Data
We configure our clinical SQLite database and hospital protocol store for the MCP server:



In [ ]:
conn = sqlite3.connect(":memory:")
cur = conn.cursor()
cur.execute("CREATE TABLE patients (mrn TEXT PRIMARY KEY, name TEXT NOT NULL, ward TEXT NOT NULL, diagnosis TEXT NOT NULL);")
cur.execute("CREATE TABLE prescriptions (id INTEGER PRIMARY KEY, mrn TEXT, medication TEXT NOT NULL, dose TEXT NOT NULL, doctor TEXT NOT NULL);")

# Seed records
cur.execute("INSERT INTO patients VALUES ('MRN882910', 'Rahul Sharma', 'Endocrinology', 'Type-2 Diabetes');")
cur.execute("INSERT INTO patients VALUES ('MRN993012', 'Priya Patel', 'Cardiology', 'Hypertension');")
cur.execute("INSERT INTO prescriptions VALUES (1, 'MRN882910', 'Metformin', '500mg PO BID', 'Dr. Smith');")
cur.execute("INSERT INTO prescriptions VALUES (2, 'MRN882910', 'Atorvastatin', '20mg Daily', 'Dr. Smith');")
cur.execute("INSERT INTO prescriptions VALUES (3, 'MRN993012', 'Lisinopril', '10mg Daily', 'Dr. Jones');")
conn.commit()

print("[+] In-memory clinical database seeded.")


---
## 2. Implementing the Python MCP Server Core
Let's build an `EnterpriseHealthcareMCPServer` that registers Tools, Resources, and Prompts:



In [ ]:
class EnterpriseHealthcareMCPServer:
    def __init__(self, db_conn):
        self.conn = db_conn
        self.tools = {}
        self.resources = {}
        self.prompts = {}

    def register_tool(self, name: str, description: str, param_schema: Dict[str, Any], handler):
        self.tools[name] = {
            "name": name,
            "description": description,
            "inputSchema": param_schema,
            "handler": handler
        }

    def register_resource(self, uri_pattern: str, name: str, handler):
        self.resources[uri_pattern] = {
            "uri_pattern": uri_pattern,
            "name": name,
            "handler": handler
        }

    def register_prompt(self, name: str, description: str, template_handler):
        self.prompts[name] = {
            "name": name,
            "description": description,
            "handler": template_handler
        }

    def handle_request(self, request: Dict[str, Any]) -> Dict[str, Any]:
        # Dispatches JSON-RPC 2.0 requests to registered handlers
        method = request.get("method")
        req_id = request.get("id")
        params = request.get("params", {})

        if method == "tools/list":
            return {
                "jsonrpc": "2.0",
                "id": req_id,
                "result": {
                    "tools": [
                        {"name": t["name"], "description": t["description"], "inputSchema": t["inputSchema"]}
                        for t in self.tools.values()
                    ]
                }
            }

        elif method == "tools/call":
            tool_name = params.get("name")
            args = params.get("arguments", {})
            if tool_name not in self.tools:
                return {"jsonrpc": "2.0", "id": req_id, "error": {"code": -32601, "message": f"Tool '{tool_name}' not found"}}
            
            try:
                content = self.tools[tool_name]["handler"](**args)
                return {
                    "jsonrpc": "2.0",
                    "id": req_id,
                    "result": {"content": [{"type": "text", "text": content}], "isError": False}
                }
            except Exception as e:
                return {
                    "jsonrpc": "2.0",
                    "id": req_id,
                    "result": {"content": [{"type": "text", "text": str(e)}], "isError": True}
                }

        elif method == "resources/read":
            uri = params.get("uri", "")
            for pattern, r_data in self.resources.items():
                match = re.match(pattern.replace("{mrn}", "([^/]+)"), uri)
                if match:
                    content = r_data["handler"](match.group(1))
                    return {
                        "jsonrpc": "2.0",
                        "id": req_id,
                        "result": {"contents": [{"uri": uri, "mimeType": "application/json", "text": content}]}
                    }
            return {"jsonrpc": "2.0", "id": req_id, "error": {"code": -32602, "message": "Resource URI not found"}}

        return {"jsonrpc": "2.0", "id": req_id, "error": {"code": -32601, "message": f"Unknown method '{method}'"}}

server = EnterpriseHealthcareMCPServer(conn)
print("[+] MCP Server engine initialized.")


---
## 3. Registering Clinical Tools, Resources & Prompts
Let's add our real tools and resources to the server:



In [ ]:
# 1. Register Tool: Query Patient Prescriptions
def tool_query_prescription(patient_name: str) -> str:
    cur = conn.cursor()
    cur.execute('''
        SELECT p.name, p.mrn, pr.medication, pr.dose, pr.doctor
        FROM patients p
        JOIN prescriptions pr ON p.mrn = pr.mrn
        WHERE LOWER(p.name) LIKE ?;
    ''', (f"%{patient_name.lower()}%",))
    rows = cur.fetchall()
    if not rows:
        return f"No records found for patient: {patient_name}"
    return json.dumps([{"patient": r[0], "mrn": r[1], "medication": r[2], "dose": r[3], "doctor": r[4]} for r in rows])

server.register_tool(
    name="query_patient_prescriptions",
    description="Fetch authorized EHR prescription records for patient.",
    param_schema={
        "type": "object",
        "properties": {"patient_name": {"type": "string"}},
        "required": ["patient_name"]
    },
    handler=tool_query_prescription
)

# 2. Register Resource: Dynamic Patient Vitals
def resource_patient_vitals(mrn: str) -> str:
    return json.dumps({
        "mrn": mrn,
        "vitals": {"blood_pressure": "120/80", "heart_rate": "72 bpm", "eGFR": "68 mL/min"},
        "status": "Stable"
    })

server.register_resource(
    uri_pattern="ehr://patients/{mrn}/vitals",
    name="Patient Vitals Stream",
    handler=resource_patient_vitals
)

# 3. Register Prompt: Clinical Reconciliation Prompt
server.register_prompt(
    name="reconcile_discharge",
    description="Standard clinical reconciliation prompt template.",
    template_handler=lambda mrn: f"Perform complete medication reconciliation for patient {mrn}."
)

print("[+] Tools, Resources, and Prompts registered on MCP Server.")


---
## 4. Testing via JSON-RPC 2.0 Dispatch
Let's test executing the MCP server via simulated JSON-RPC calls:



In [ ]:
# Test 1: Discover Tools
req_list = {"jsonrpc": "2.0", "id": 101, "method": "tools/list"}
res_list = server.handle_request(req_list)
print("Tools List Count:", len(res_list["result"]["tools"]))

# Test 2: Execute Tool
req_call = {
    "jsonrpc": "2.0",
    "id": 102,
    "method": "tools/call",
    "params": {"name": "query_patient_prescriptions", "arguments": {"patient_name": "Rahul"}}
}
res_call = server.handle_request(req_call)
print("
Tool Call Result:")
print(res_call["result"]["content"][0]["text"])

# Test 3: Read Resource
req_res = {
    "jsonrpc": "2.0",
    "id": 103,
    "method": "resources/read",
    "params": {"uri": "ehr://patients/MRN882910/vitals"}
}
res_res = server.handle_request(req_res)
print("
Resource Read Result:")
print(res_res["result"]["contents"][0]["text"])


---
## 💡 Summary
- You built a functional **MCP Server** implementing the exact JSON-RPC 2.0 protocol.
- In **Lab 3**, we will explore **MCP Security Threats & Defenses**!

